In [ ]:
# =====================================================
# SETUP
# =====================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

In [ ]:
# =====================================================
# CREATE DATASET
# =====================================================

data = [
("This phone is amazing",1),
("I love this laptop",1),
("Battery life is terrible",0),
("Worst product ever",0),
("The camera quality is great",1),
("The screen broke in two days",0),
("Excellent performance",1),
("Very slow and frustrating",0),
("The movie is fantastic",1),
("The movie is boring",0),
("Great design and build quality",1),
("The software crashes often",0)
]

df = pd.DataFrame(data, columns=["text","label"])
df

,text,label
0,This phone is amazing,1
1,I love this laptop,1
2,Battery life is terrible,0
3,Worst product ever,0
4,The camera quality is great,1
5,The screen broke in two days,0
6,Excellent performance,1
7,Very slow and frustrating,0
8,The movie is fantastic,1
9,The movie is boring,0


In [ ]:
# =====================================================
# TF-IDF FEATURE EXTRACTION
# =====================================================

vectorizer = TfidfVectorizer()

X = vectorizer.fit_transform(df["text"])
y = df["label"]

print("Vocabulary:", vectorizer.get_feature_names_out())
print("Feature shape:", X.shape)

Vocabulary: ['amazing' 'and' 'battery' 'boring' 'broke' 'build' 'camera' 'crashes'
 'days' 'design' 'ever' 'excellent' 'fantastic' 'frustrating' 'great' 'in'
 'is' 'laptop' 'life' 'love' 'movie' 'often' 'performance' 'phone'
 'product' 'quality' 'screen' 'slow' 'software' 'terrible' 'the' 'this'
 'two' 'very' 'worst']
Feature shape: (12, 35)


In [ ]:
# =====================================================
# TRAIN MODEL
# =====================================================

model = LogisticRegression()

model.fit(X,y)

pred = model.predict(X)

print(classification_report(y,pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00         6
           1       1.00      1.00      1.00         6

    accuracy                           1.00        12
   macro avg       1.00      1.00      1.00        12
weighted avg       1.00      1.00      1.00        12



In [ ]:
# =====================================================
# TEST NEW SENTENCE
# =====================================================

test = ["movie is pathetic"]

test_vec = vectorizer.transform(test)

prediction = model.predict(test_vec)

print("Prediction:",prediction)

Prediction: [1]


In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 60.7 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

import gensim.downloader as api

In [ ]:
# =====================================================
# LOAD PRETRAINED WORD2VEC
# =====================================================

w2v_model = api.load("word2vec-google-news-300")

print("Vector size:", w2v_model.vector_size)

[==================================================] 100.0% 1662.8/1662.8MB downloaded
Vector size: 300


In [ ]:
w2v_model.most_similar("movie")

[('film', 0.8676770329475403),
 ('movies', 0.8013108372688293),
 ('films', 0.7363011837005615),
 ('moive', 0.6830360889434814),
 ('Movie', 0.6693680286407471),
 ('horror_flick', 0.6577848792076111),
 ('sequel', 0.6577793955802917),
 ('Guy_Ritchie_Revolver', 0.650975227355957),
 ('romantic_comedy', 0.6413198709487915),
 ('flick', 0.6321909427642822)]

In [ ]:
# =====================================================
# CONVERT SENTENCE TO VECTOR
# =====================================================

def sentence_vector(sentence):

    words = sentence.lower().split()

    vectors = []

    for word in words:
        if word in w2v_model:
            vectors.append(w2v_model[word])

    if len(vectors) == 0:
        return np.zeros(300)

    return np.mean(vectors, axis=0)

In [ ]:
# =====================================================
# BUILD FEATURE MATRIX
# =====================================================

X = np.vstack(df["text"].apply(sentence_vector))
y = df["label"]

print("Feature shape:", X.shape)

Feature shape: (12, 300)


In [ ]:
# =====================================================
# TRAIN MODEL
# =====================================================

model = LogisticRegression()

model.fit(X,y)

pred = model.predict(X)

print(classification_report(y,pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00         6
           1       1.00      1.00      1.00         6

    accuracy                           1.00        12
   macro avg       1.00      1.00      1.00        12
weighted avg       1.00      1.00      1.00        12



In [ ]:
# =====================================================
# TEST NEW SENTENCES
# =====================================================

test_sentences = [
"This phone is crashing",
"This product is awesome",
"The movie is pathetic"
]

X_test = np.vstack([sentence_vector(s) for s in test_sentences])

predictions = model.predict(X_test)

for text, pred in zip(test_sentences, predictions):
    print(text, "→", "Positive" if pred==1 else "Negative")

This phone is crashing → Negative
This product is awesome → Positive
The movie is pathetic → Negative


In [ ]:
s1 = "dog bites man"
s2 = "man bites dog"

v1 = sentence_vector(s1)
v2 = sentence_vector(s2)

print("Vectors equal:", np.allclose(v1, v2))

Vectors equal: True


In [ ]:
# =====================================================
# INSTALL TRANSFORMERS
# =====================================================

!pip install transformers torch

In [ ]:
# =====================================================
# LOAD PRETRAINED TRANSFORMER
# =====================================================

from transformers import pipeline

sentiment_model = pipeline("sentiment-analysis")

print("Model loaded successfully")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Model loaded successfully


In [ ]:
# =====================================================
# BASIC SENTIMENT TEST
# =====================================================

sentences = [
"This phone is amazing",
"This product is terrible",
"I love this movie",
"I hate this movie"
]

for s in sentences:
    print(s, "→", sentiment_model(s))

This phone is amazing → [{'label': 'POSITIVE', 'score': 0.9998575448989868}]
This product is terrible → [{'label': 'NEGATIVE', 'score': 0.9997355341911316}]
I love this movie → [{'label': 'POSITIVE', 'score': 0.9998766183853149}]
I hate this movie → [{'label': 'NEGATIVE', 'score': 0.9996687173843384}]


In [ ]:
# =====================================================
# WORD ORDER DEMO
# =====================================================

sentences = [
"dog bites man",
"man bites dog"
]

for s in sentences:
    print(s, "→", sentiment_model(s))

dog bites man → [{'label': 'NEGATIVE', 'score': 0.9866145253181458}]
man bites dog → [{'label': 'NEGATIVE', 'score': 0.9722158312797546}]


In [ ]:
# =====================================================
# LOAD TRANSFORMER MODEL
# =====================================================

from transformers import AutoTokenizer, AutoModel
import torch

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# =====================================================
# SENTENCE EMBEDDING FUNCTION
# =====================================================

def sentence_embedding(sentence):

    inputs = tokenizer(sentence, return_tensors="pt")

    with torch.no_grad():
        outputs = model(**inputs)

    cls_embedding = outputs.last_hidden_state[:,0,:]

    return cls_embedding.squeeze().numpy()

In [ ]:
# =====================================================
# WORD ORDER DEMONSTRATION
# =====================================================

s1 = "dog bites man"
s2 = "man bites dog"

v1 = sentence_embedding(s1)
v2 = sentence_embedding(s2)

print("Vector distance:", torch.norm(torch.tensor(v1) - torch.tensor(v2)))

Vector distance: tensor(2.1677)
